# Octavo

> Build nbdev documentation with Octavo
- order: 21

In [ ]:
#| default_exp octavo

#| export 
Octavo is an alternative to Quarto for building a documentation website from nbdev notebooks. Install `nbdev[octavo]` to use it. A project selects Octavo by placing `octavo.yml` in its configured notebooks directory. Projects without that file continue to use Quarto and do not import Octavo.

The usual `nbdev-docs` command builds the site. Octavo currently builds all pages together. File-selection options are rejected unless left at their defaults; `n_workers` affects only Quarto builds.

Run `nbdev-export` before building to update the generated Python modules and symbol indexes. `nbdev-readme` and `nbdev-contributing` also use Octavo for these projects. `nbdev-prepare` still exports, tests, and cleans notebooks, but skips Quarto configuration generation.

For an existing Quarto project, `nbdev-migrate-octavo` converts documentation sources in place. The converted sources are not intended to work unchanged with both builders. Use `nbdev-preview` to serve the site and rebuild after edits. Refresh the browser manually.

## Migration instructions (share this with your AI)

Octavo builds into the same output directory as Quarto and replaces what's there. The migration also converts the sources in place, and they won't build with Quarto afterwards, even if you keep `_quarto.yml`. So save the Quarto site first:

1. Commit or back up the project.
2. From the project root, run `nbdev-docs` while the project still uses Quarto. Check that the build works, and note any failures it already has, so you don't mistake them for migration problems later.
3. Copy the whole built site, including assets and search files, to a directory outside the project's source and output directories. This copy is the baseline you compare against.
4. Run `nbdev-migrate-octavo` from the project root. Read its changes and everything it prints: it reports the settings it couldn't carry over.
5. Run `nbdev-docs` again to build with Octavo. If the build fails, keep its output for diagnosis.

`nbdev-migrate-octavo` renames `.qmd` sources to `.md`, updates references to them, converts directives, and moves a page's `subtitle` to `description`. Octavo reads Quarto's other front matter and settings names as they are. The migration writes `octavo.yml` from `_quarto.yml` and `nbdev.yml`, and leaves an existing `octavo.yml` alone. It doesn't port the site's design. It leaves stylesheet files unchanged, and it doesn't add the stylesheets listed under `format.html.css` in `_quarto.yml` to `octavo.yml`. Stylesheets that a page lists in its own front matter still load.

Read the original `_quarto.yml`, `nbdev.yml` and `pyproject.toml`, and the new `octavo.yml`. Go through the checks below that apply to the project, skipping features it doesn't use, and propose a short migration plan. Ask me before accepting anything that loses behavior.

- **Styles and custom pages.** Octavo's HTML has no Bootstrap or Quarto classes. Look at the rendered Octavo HTML before reusing old CSS, and copy over only the rules that still apply. Don't enable an old stylesheet wholesale or add a compatibility layer. Check narrow screens too.
- **Wide content.** Octavo has no equivalent of Quarto's `page-layout: full` or column classes such as `.column-page` and `.column-screen-inset-right`. Widen those pages with your own CSS, in a stylesheet listed in the page's `css` front matter.
- **Layouts and tables of contents.** Octavo reads Quarto's `page-layout`, `toc` and `toc-depth` with the same meanings, but of the `page-layout` values it supports only `custom`, for a hand-built page. Check pages that used `article` or `full`, and compare tables of contents with the baseline on ordinary pages as well as custom ones.
- **Sidebar.** If the project has a hand-written `sidebar.yml` (`custom_sidebar` in `pyproject.toml`), the migration turns it into `sidebar-links` in `octavo.yml`. Compare its sections and order with the baseline. Add the pages behind any entries it reported as dropped, such as `auto` or globs, as explicit links.
- **Scripts.** The migration doesn't carry over scripts that Quarto added through `include-in-header`, `include-after-body` or raw HTML. Load them with Octavo's `js` setting, in `octavo.yml` for the whole site or in a page's front matter. Octavo loads them as module scripts (`<script type="module">`).
- **Maths.** The migration sets `math: dollars`, Quarto's `$...$` syntax. Check the pages with maths. A literal dollar sign in a paragraph that also has maths needs escaping as `\$`.
- **Conditional and embedded content.** Compare `when-format` and `unless-format` blocks with the originals, in both the HTML and the Markdown output. The conversion doesn't always keep exclusion conditions, and Markdown inside an HTML-only block isn't rendered. Where needed, turn HTML-only content into a raw `{=html}` block. Check each result rather than rewriting every block by pattern.
- **`{python}` blocks.** Octavo runs the `{python}` blocks in `.md` files on every build, and the page shows only their output, not the code. Quarto also showed the code. Compare pages that have these blocks with the baseline. If the project set `execute: enabled: false` in `_quarto.yml`, the blocks didn't run under Quarto at all.
- **Shortcodes.** Octavo doesn't support Quarto shortcodes such as `{{< include file.md >}}` or `{{< video url >}}`. They appear as literal text. Search the sources for `{{<` and report each use.
- **Render scripts and processors.** If the project has render scripts (such as `page.md.py` or `drawing.svg.py`) or custom nbdev processors, check the content and assets they produce.
- **Code that finds docs by name.** Search the whole project for `.qmd`, including build scripts, tests, generators and contributor docs, and update paths, globs and extension checks. A glob that matches nothing fails silently. Run the full test suite, not only `nbdev-docs`, and check that tests that read doc pages still find them. Regenerate saved notebook outputs that contain `.qmd` links.
- **Published files.** Quarto published only linked files and declared resources. Octavo publishes every visible file in the notebooks directory (no part of its path starts with `.` or `_`), as a page or as a copied file. Compare the output file list with the baseline. To keep stray files such as READMEs or notes out, add glob patterns to `exclude` in `octavo.yml`. Patterns are relative to the notebooks directory, and a pattern that matches a directory excludes everything in it. Don't exclude Quarto-only files: they're deleted at the end.
- **Navigation and URLs.** Check navbar links, dropdowns, icons, favicon, footer and existing public URLs. Restore icons the migration dropped, and keep the existing footer. Markdown copies of notebook pages move from `page.html.md` to `page.md`, and Octavo doesn't redirect the old addresses. Report this rather than working around it.
- **Settings that weren't carried over.** List the site and page settings the migration reported as having no Octavo equivalent, and any theme options and preview preferences. Octavo uses a page's `image` front matter as its link preview image when `site-url` is set, and a page can set its own `open-graph` or `twitter-card` options. It doesn't pick a preview image from the page content, so list the pages that relied on that. For each setting, propose a replacement or say it's dropped. Quarto's repo-actions links, such as "Report an issue", are dropped on purpose: don't list them.
- **Build hooks.** If the project has custom build hooks, keep the outputs they produce with nbdev's `docs_post_build` setting, and check the links in any indexes they generate.
- **Output directory.** A relative `output` in `octavo.yml` is relative to the notebooks directory. Without it, nbdev uses `doc_path`, which is relative to the project. Check that this is the directory the deployment workflow publishes.

Once I've approved the plan, make the changes in small steps, running `nbdev-export` before each build. Then:

- run `nbdev-docs` and the project's other Markdown commands, such as `nbdev-readme`;
- compare the HTML, Markdown, assets and other build outputs with the baseline, and check a few API signatures, symbol links and GitHub source links;
- sort the differences into failures the baseline already had, expected changes, and regressions.

A clean build doesn't show that the site works in a browser. Serve the baseline and the new site separately, and check that each server points at the directory you meant. Serve the new site under its real subpath, such as `/project/`, not only at `/`. Then, on a few representative pages at desktop and phone widths, test:

- navigation in the menu's
- anchors and images;
- search, following results from the homepage and from a nested page;
- the optional features you found in the checks above.

Report what you changed, which checks you actually ran, the differences you couldn't resolve, the settings that weren't carried over, and the decisions I need to make. Keep the original configuration and the baseline until I've reviewed the migration.

After the review, delete `_quarto.yml`, plus the files it lists under `metadata-files` and `syntax-definitions` that nothing else in the project uses, such as `nbdev.yml` and `sidebar.yml`.

In [ ]:
#| export
from contextlib import redirect_stdout
from io import StringIO
from fastcore.utils import *
from fastcore.script import call_parse
import yaml, shutil, subprocess, threading, webbrowser
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from octavo import *
from octavo.migrate import *
from mdhtml import md2gfm
from nbdev.config import get_config, import_obj, _load_toml
from nbdev.doclinks import NbdevLookup
from nbdev.process import NBProcessor
from nbdev.processors import FilterDefaults, add_links
from nbdev.serve import _is_qpy
from nbdev.quarto import _doc_mtime_not_older, fs_watchdog

In [ ]:
#| hide
import tempfile
from copy import deepcopy
from fastcore.test import *
from fastcore.nbio import new_nb, mk_cell, read_nb, write_nb
from nbdev.quarto import nbdev_docs

## Notebook processing

Notebooks still need nbdev’s documentation processing before Octavo renders them. `OctavoProc` reuses that processing, including the project’s `doc_procs` and `skip_procs` settings.

It keeps directives such as `#| echo: false` for Octavo to interpret. It also places generated page metadata, such as the title, in a Markdown cell that Octavo can read. The caller chooses whether generated symbol links point into the current site or to published documentation. Links written explicitly by the author are unchanged.

In [ ]:
#| export
class OctavoProc(FilterDefaults):
    "Prepare notebook content, metadata, and symbol links for Octavo"
    def __init__(self, local_lib=None, strip_libs=None): self.nl = NbdevLookup(strip_libs=strip_libs or local_lib, local_lib=local_lib)
    def add_links(self, cell): return add_links(cell, nl=self.nl)
    def procs(self): return L(self.add_links if o is add_links else o for o in super().procs())
    def nb_proc(self, nb): return NBProcessor(nb=nb, procs=self.procs(), rm_directives=False)

    def __call__(self, nb):
        super().__call__(nb)
        if getattr(nb, 'frontmatter_', None) and nb.cells[0].cell_type=='raw': nb.cells[0].cell_type = 'markdown'

In [ ]:
source = new_nb([mk_cell('# Example\n\n> A notebook with rendering directives', 'markdown'),
                 mk_cell('Use `ModuleMaker`. See [production docs](https://nbdev.fast.ai/api/maker.html#modulemaker).', 'markdown'),
                 mk_cell('#| echo: false\n1 + 1')])
source.path_ = Path('example.ipynb')
nb = deepcopy(source)
OctavoProc(local_lib='nbdev')(nb)
test_eq(nb.cells[0].cell_type, 'markdown')
text = '\n'.join(c.source for c in nb.cells)
assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in text
assert '[production docs](https://nbdev.fast.ai/api/maker.html#modulemaker)' in text
assert '#| echo: false' in text
quarto_nb = deepcopy(source)
FilterDefaults()(quarto_nb)
test_eq(quarto_nb.cells[0].cell_type, 'raw')
assert '#| echo: false' in '\n'.join(c.source for c in quarto_nb.cells)
nb.cells[0].source

'---\ndescription: A notebook with rendering directives\noutput-file: example.html\ntitle: Example\n\n---\n\n'

A website build needs to distinguish this project’s API from dependencies’ APIs. References to this project should lead to pages in the site being built; references to dependencies should lead to their published documentation.

nbdev identifies the project’s symbol indexes through `[project.entry-points.nbdev]` in `pyproject.toml`. It cannot infer those names from the package’s installation name. In the example below, the distribution is named `different-distribution`, but its symbol index is registered as `nbdev`.

In [ ]:
#| export
def local_libraries(cfg):
    "Names of the project's declared nbdev symbol indexes"
    proj = _load_toml(cfg.config_file).get('project', {})
    return tuple(proj.get('entry-points', {}).get('nbdev', {}))

In [ ]:
with tempfile.TemporaryDirectory() as d:
    project = Path(d)
    (project/'pyproject.toml').write_text('[project]\nname = "different-distribution"\n[project.entry-points.nbdev]\nnbdev = "nbdev._modidx:d"\n[tool.nbdev]\n')
    libs = local_libraries(get_config(project))
    test_eq(libs, ('nbdev',))
    lookup = NbdevLookup(local_lib=libs)
    assert lookup.doc('nbdev.maker.ModuleMaker').startswith('/api/maker.html#')
    assert NbdevLookup().doc('nbdev.maker.ModuleMaker').startswith('https://')
    nb = deepcopy(source)
    with working_directory(project): OctavoProc(local_lib=libs)(nb)
    assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in '\n'.join(c.source for c in nb.cells)

/Users/rensdimmendaal/aai/ws/execnb/execnb/shell.py:172: UserWarning: Failed loading extension dialoghelper.bootstrap: Local skills already enabled for /Users/rensdimmendaal/aai/ws/nbdev/nbs/api
  for w in warns: warnings.warn(w)


## Render scripts

Render scripts generate documentation content during a build. Their filenames specify what they produce: `page.md.py` prints Markdown for a page, while `drawing.svg.py` prints an SVG image.

nbdev executes each script and captures its printed output. Generated Markdown goes through Octavo’s page renderer. Other output is written verbatim under the filename without `.py`; for example, `drawing.svg.py` produces `drawing.svg`.

A `.md.py` script can put YAML page metadata in its opening docstring. nbdev prepends that metadata to the printed Markdown.

Render-script source files are not published. Ordinary single-suffix `.py` files are copied as assets rather than executed. An exception in a render script stops the build.

Scripts under paths matched by the `exclude` setting in `octavo.yml` are not run.

In [ ]:
#| export
def render_script(path):
    "Execute a double-suffix render script and return its printed output with optional page metadata"
    path = Path(path)
    if path.suffix!='.py' or len(path.suffixes)<2: return
    fm = _is_qpy(path) if path.name.endswith('.md.py') else None
    f = StringIO()
    with redirect_stdout(f): exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), {})
    return (f'---\n{fm}\n---\n\n' if fm else '') + f.getvalue()

In [ ]:
#| export
def script_pages(root,         # Documentation directory
                 exclude=None): # Glob patterns of paths to skip, as in Octavo's `exclude` setting
    "Render scripts under `root`: `*.md.py` as `Page`s, anything else as `GeneratedFile`s"
    root,pages,files = Path(root),L(),L()
    for rel in source_files(root, exclude).map(Path).filter(lambda o: o.suffix=='.py'):
        if (txt:=render_script(root/rel)) is None: continue
        dest = rel.with_suffix('')
        if dest.suffix=='.md': pages.append(Page(str(rel), txt, nm=page_nm(dest)))
        else: files.append(GeneratedFile(str(rel), str(dest), txt))
    return pages,files

In [ ]:
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d/'01_intro.md.py').write_text('"""---\ntitle: Hi\n---"""\nprint("# Generated")\nprint("body")')
    (d/'images').mkdir(); (d/'images/dot.svg.py').write_text('print("<svg/>")')
    (d/'plain.md.py').write_text('print("# No frontmatter")')
    (d/'ordinary.py').write_text('x = 1')
    (d/'_skip').mkdir(); (d/'_skip/no.md.py').write_text('"""---\n---"""\nprint("no")')
    pages,files = script_pages(d)
    test_eq([p.dest for p in pages], ['intro.html','plain.html'])
    test_eq(pages[0].text, '---\ntitle: Hi\n---\n\n# Generated\nbody\n')
    test_eq(pages[0].meta['title'], 'Hi')
    test_eq(pages[1].text, '# No frontmatter\n')
    test_eq([(f.src,f.dest,f.content) for f in files], [('images/dot.svg.py','images/dot.svg','<svg/>\n')])
    test_eq(script_pages(d, exclude=['images'])[1], [])
    test_is(render_script(d/'ordinary.py'), None)
    (d/'bad.md.py').write_text('1/0')
    with expect_fail(ZeroDivisionError, contains='division'): script_pages(d)

## Post-build callback

Some projects publish extra files alongside their documentation, such as an API list or an LLM context file. Use a post-build callback to generate these after the website has been built.

Set `docs_post_build` to an importable `module:function` in `pyproject.toml`:

```toml
[tool.nbdev]
docs_post_build = "my_project.docbuild:build_extras"
```

After a successful site build, nbdev calls the function once with `(out, cfg)`. `out` is the absolute directory containing the completed site. `cfg` is the nbdev project configuration. The callback writes additional files into `out`; its return value is ignored. Callback errors fail the build. Without this setting, no callback runs.

Author the callback in a notebook and export it before building the docs, as with custom processors. Its dependencies belong to the project's documentation setup.

## Build configuration

`octavo.yml` contains named settings, such as `title: Demo`, read by Octavo's `read_cfg`. An empty file selects Octavo’s defaults. A YAML list or a single value such as `false` cannot supply those settings.

The build checks this file before running render scripts or notebook processors. Invalid configuration therefore fails before project code executes.

In [ ]:
#| export
def octavo_config(root):
    "Settings from `octavo.yml`, as Octavo's `read_cfg` reads them; the file must exist"
    if not (p := Path(root)/CFG_FILE).is_file(): raise FileNotFoundError(p)
    return read_cfg(root)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    test_fail(lambda: octavo_config(root), contains='octavo.yml')
    for text in ('- not a mapping', '[]', 'false', '0'):
        (root/'octavo.yml').write_text(text)
        test_fail(lambda: octavo_config(root), contains='mapping')
    (root/'octavo.yml').write_text('')
    test_eq(octavo_config(root), {})

Links to this project’s API should stay within the website being built. For example, when building nbdev’s website, a generated link for `ModuleMaker` points to `/api/maker.html#modulemaker` rather than the published nbdev site.

Links to dependencies still point to their published documentation. An absolute URL written by the author is not changed.

In [ ]:
#| export
def octavo_docs(
    path:str=None, # Path to the project or its notebooks
):
    "Create Octavo docs and run the optional docs_post_build callback"
    cfg = get_config(path)
    settings = {'output': cfg.doc_path, **octavo_config(cfg.nbs_path)}
    with working_directory(cfg.config_path):
        pages,files = script_pages(cfg.nbs_path, settings.get('exclude'))
        docs = scan(cfg.nbs_path, nbproc=OctavoProc(local_lib=local_libraries(cfg)), exclude=settings.get('exclude')) + pages
        site = Site(docs, root=cfg.nbs_path, files=files, **settings)
        if (miss := site.build()): print(miss)
        if (post := cfg.get('docs_post_build')):
            out = Path(site.settings['output']).expanduser()
            if not out.is_absolute(): out = cfg.nbs_path/out
            import_obj(post)(out.resolve(), cfg)

## Build a documentation site

`nbdev-docs` reads `octavo.yml` and processes notebooks and render scripts together. It does not migrate sources or change `_quarto.yml`. This temporary project builds linked pages and generated assets, then demonstrates that invalid configuration is rejected before executing a render script.

In [ ]:
build_tmp = tempfile.TemporaryDirectory()
project = Path(build_tmp.name)
nbs = project/'nbs'
nbs.mkdir()
(project/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "nbs"\nskip_procs = "insert_warning"\n')
(nbs/'octavo.yml').write_text('title: Demo\noutput: ../_octavo\nsite-url: https://example.com\n')
(nbs/'_quarto.yml').write_text('project:\n  type: website\n')
(nbs/'index.md').write_text('# Home\n\nRead the [guide](guide.ipynb).')
guide = new_nb([mk_cell('# Guide\n\n> A notebook page', 'markdown'), mk_cell('Notebook content.', 'markdown')])
write_nb(guide, nbs/'guide.ipynb')
(nbs/'generated.md.py').write_text('print("# Generated page")')
(nbs/'image.svg.py').write_text('print("<svg/>")')

15

The ordinary build command leaves Quarto configuration unchanged. Without `output` in `octavo.yml`, it uses nbdev’s project-relative `doc_path`. An explicit Octavo output takes precedence and is relative to the notebooks directory. Here, `../_octavo` overrides the default project-level `_docs` directory.

In [ ]:
with working_directory(project): run('nbdev-docs')
out = project/'_octavo'
assert 'guide.html' in (out/'index.html').read_text()
assert 'Notebook content.' in (out/'guide.html').read_text()
assert 'AUTOGENERATED' not in (out/'guide.md').read_text()
assert (out/'generated.html').exists()
test_eq((out/'image.svg').read_text(), '<svg/>\n')
test_eq((nbs/'_quarto.yml').read_text(), 'project:\n  type: website\n')

Existing scripts can pass the default file-selection options explicitly. Octavo accepts those calls, but still builds the whole site. For normal use, run `nbdev-docs` without file-selection options.

In [ ]:
with working_directory(project): run(r'nbdev-docs --file-re "\.(?:ipynb|qmd|html)$"')
nbdev_docs(project, file_glob=None)

/Users/rensdimmendaal/aai/ws/execnb/execnb/shell.py:172: UserWarning: Failed loading extension dialoghelper.bootstrap: Local skills already enabled for /Users/rensdimmendaal/aai/ws/nbdev/nbs/api
  for w in warns: warnings.warn(w)


The Octavo integration currently builds the whole site. Non-default file filters are rejected because selective builds are not implemented.

Invalid configuration must fail before any render script runs.

In [ ]:
test_fail(lambda: nbdev_docs(project, file_glob='*.ipynb'), contains='file_glob')
(nbs/'octavo.yml').write_text('- invalid')
(nbs/'generated.md.py').write_text('raise RuntimeError("script ran before validation")')
test_fail(lambda: nbdev_docs(project), contains='mapping')
build_tmp.cleanup()

## Local preview

`nbdev-preview` builds and serves an Octavo site at `http://localhost:8000`. Use `--port`, `--host`, and `--no-browser` to change the server options. It watches the project for edits and rebuilds the whole site in a fresh process. Refresh the browser manually after each build.

Preview writes directly to the configured documentation output, unlike Quarto’s separate preview cache. Pages can be unavailable during a rebuild. A failed build clears partial output and shows an error page; the server keeps watching so the next successful build restores the site. Errors appear in the terminal.

Hidden paths, Python bytecode caches, and the output directory are ignored. Restart preview after changing the notebooks or output directory. Octavo always builds on startup; `--render` is redundant and `--n-workers` remains Quarto-only.

In [ ]:
#| export
def octavo_preview(
    path:str=None, # Path to the project or its notebooks
    port:int=None, # HTTP port (defaults to 8000)
    host:str=None, # Bind address (defaults to localhost)
    no_browser:bool=False, # Do not open a browser
):
    "Serve documentation and rebuild when project files change"
    cfg = get_config(path)
    root = cfg.config_path.resolve()
    settings = octavo_config(cfg.nbs_path)
    out = (cfg.nbs_path/Path(settings.get('output', cfg.doc_path)).expanduser()).resolve()
    if root.is_relative_to(out) or cfg.nbs_path.resolve().is_relative_to(out): raise ValueError('Output must not contain sources')
    changed = threading.Event()

    def rebuild():
        try: subprocess.run(['nbdev-docs', '--path', str(root)], cwd=root, check=True)
        except subprocess.CalledProcessError:
            shutil.rmtree(out, ignore_errors=True)
            out.mkdir(parents=True, exist_ok=True)
            (out/'index.html').write_text('<h1>Build failed - see terminal</h1>')
            print('Build failed; waiting for another edit.', flush=True)

    def on_change(event):
        if event.event_type not in ('created', 'modified', 'deleted', 'moved'): return
        if event.is_directory and event.event_type=='modified': return
        for name in (event.src_path, getattr(event, 'dest_path', '')):
            if not name: continue
            src = Path(name).resolve()
            if src.is_relative_to(out) or not src.is_relative_to(root): continue
            if any(p.startswith('.') or p=='__pycache__' for p in src.relative_to(root).parts): continue
            changed.set()

    handler = partial(SimpleHTTPRequestHandler, directory=str(out))
    with ThreadingHTTPServer((host or 'localhost', 8000 if port is None else port), handler) as server:
        url = f'http://{host or "localhost"}:{server.server_port}'
        with fs_watchdog(on_change, root):
            rebuild()
            threading.Thread(target=server.serve_forever, daemon=True).start()
            try:
                print(f'{url} - refresh your browser after rebuilding', flush=True)
                if not no_browser: webbrowser.open(url)
                while True:
                    changed.wait()
                    changed.clear()
                    while changed.wait(0.2): changed.clear()
                    rebuild()
            finally: server.shutdown()

## Repository Markdown

`nbdev-readme` and `nbdev-contributing` render the configured notebooks to the project root without Quarto. They use stored outputs and do not execute ordinary code cells. nbdev processors still run, including custom processors and generated API documentation. Titles and descriptions become Markdown. Symbol links use published documentation URLs.

Images stored in notebook outputs are extracted into a directory beside the generated Markdown. The directory uses the source notebook’s name: `index.ipynb` produces `index_files`, even though the generated document is named `README.md`. This matches nbdev’s Quarto convention.

Authored relative links are copied unchanged; their destinations are not adjusted for the document’s new location.

A missing source notebook is skipped. With `chk_time=True`, an existing document is skipped when its modification time is at least as recent as the source notebook’s.

In [ ]:
#| export
def octavo_readme(
    path:str=None, # Path to the project or its notebooks
    chk_time:bool=False, # Only build if the source is newer
    contributing:bool=False, # Render CONTRIBUTING instead of README
):
    "Render a repository Markdown document from a notebook's stored outputs"
    cfg = get_config(path)
    nbname = cfg.get('contributing_nb', 'contributing.ipynb') if contributing else cfg.readme_nb
    root = Path(path).absolute() if path else cfg.nbs_path
    src = root/nbname
    if not src.exists(): src = cfg.nbs_path/nbname
    if not src.exists(): return
    name = 'CONTRIBUTING' if contributing else 'README'
    dest = cfg.config_path/f'{name}.md'
    if chk_time and _doc_mtime_not_older(dest, src): return
    octavo_config(cfg.nbs_path)
    with working_directory(cfg.config_path):
        proc = OctavoProc(strip_libs=local_libraries(cfg))
        doc = md2doc(ipynb2md(src, nbproc=proc))
        head = f'# {doc.meta["title"]}\n\n' if doc.meta.get('title') else ''
        if desc:=doc.meta.get('description'): head += f'{desc}\n\n'
        md2gfm(head + doc.body, dest=dest, imgdir=cfg.config_path/f'{src.stem}_files', raw=('md', 'html'))

In [ ]:
readme_tmp = tempfile.TemporaryDirectory()
project = Path(readme_tmp.name)
nbs = project/'nbs'
nbs.mkdir()
(project/'pyproject.toml').write_text(r'''[project]
name = "different-distribution"
[project.entry-points.nbdev]
nbdev = "nbdev._modidx:d"
[tool.nbdev]
nbs_path = "nbs"
contributing_nb = "help.ipynb"
''')
(nbs/'octavo.yml').write_text('{}')

2

README generation should use a notebook’s saved results without rerunning its ordinary code cells. The example below constructs a notebook containing a saved image and code that raises an exception if executed.

Successful rendering must include the saved image without running that code. This matches Quarto’s `--no-execute` behaviour.

In [ ]:
png = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+aX1sAAAAASUVORK5CYII='
image = mk_cell('#| echo: false\nraise RuntimeError("must not execute")')
image.outputs = [dict(output_type='display_data', metadata={}, data={'image/png': png})]
book = new_nb([mk_cell('# Example\n\n> Repository document', 'markdown'),
               mk_cell('Use `ModuleMaker`. [Guide](guide.md). [Reference][ref]\n\n[ref]: guide.md', 'markdown'),
               image, mk_cell('#| include: false\n"excluded cell"')])
write_nb(book, nbs/'index.ipynb')
write_nb(book, nbs/'help.ipynb')

README and CONTRIBUTING use separate source notebooks and produce separate Markdown files. This test gives both notebooks identical content to check that both commands handle hidden code, saved images, and links consistently.

In [ ]:
cwd = Path.cwd()
octavo_readme(project)
with working_directory(project):
    run('nbdev-readme')
    run('nbdev-contributing')

/Users/rensdimmendaal/aai/ws/execnb/execnb/shell.py:172: UserWarning: Failed loading extension dialoghelper.bootstrap: Local skills already enabled for /Users/rensdimmendaal/aai/ws/nbdev/nbs/api
  for w in warns: warnings.warn(w)


A README is read in the repository, not alongside the generated API pages. Its generated symbol links therefore point to published documentation—even when they refer to this project’s own API.

For example, nbdev’s README links `ModuleMaker` to the published nbdev website. The nbdev website itself uses a local API link. This difference depends on where the document will be read, not on which library the symbol belongs to.

In [ ]:
for name,stem in (('README', 'index'), ('CONTRIBUTING', 'help')):
    text = (project/f'{name}.md').read_text()
    assert '# Example' in text and 'Repository document' in text
    assert 'https://nbdev.fast.ai/api/maker.html#modulemaker' in text
    assert '(guide.md)' in text and '[ref]: guide.md' in text
    assert 'must not execute' not in text and 'excluded cell' not in text
    assert 'data:image' not in text and f'{stem}_files/' in text
    assert list((project/f'{stem}_files').glob('*.png'))

An up-to-date README should not be rewritten. Projects without a CONTRIBUTING source should still build their other documents.

In [ ]:
before = (project/'README.md').stat().st_mtime_ns
octavo_readme(project, chk_time=True)
test_eq((project/'README.md').stat().st_mtime_ns, before)
assert not (nbs/'_quarto.yml').exists()
(nbs/'help.ipynb').unlink()
octavo_readme(project, contributing=True)
readme_tmp.cleanup()

## Source migration

`nbdev-migrate-octavo` converts Quarto documentation sources in place. Commit or back up the project before running it. The command:

1. Renames `.qmd` documents to `.md` and `.qmd.py` render scripts to `.md.py`.
2. Rewrites references to renamed documents, and to Markdown copies of pages: Quarto named the copy of `page.ipynb` `page.html.md`, while Octavo names it `page.md`. This includes plain-text files such as `llms.txt`.
3. Replaces Quarto content-visible block markers with raw-format fences.
4. Moves `subtitle` page metadata to `description`. Octavo reads Quarto's other metadata names as they are.
5. Creates `octavo.yml` from selected Quarto settings when no Octavo configuration exists.
6. Reports the page and site settings that Octavo has no equivalent for.
7. Leaves `_quarto.yml` unchanged for reference.

The presence of `octavo.yml` selects Octavo.

Migration converts sources and configuration, not the site’s design. CSS, layouts, and themes need project-specific review. Stylesheet files remain unchanged, and of Quarto’s `page-layout` values Octavo supports only `custom`. Quarto stylesheet references are not copied into `octavo.yml`.

The source conversions and unsupported-setting reports live in `octavo.migrate`. nbdev owns render-script discovery, renaming and YAML-docstring conversion, and supplies project configuration, the output directory override, custom-sidebar selection, and its accepted metadata keys. The examples below exercise that integration.

Configuration is parsed before any source files are changed. Empty YAML uses defaults; a scalar or list cannot represent site settings.

In [ ]:
#| export
def _quarto_config(nbs_path): return read_quarto_config(nbs_path/'_quarto.yml'),read_quarto_config(nbs_path/'nbdev.yml')

nbdev already knows where to put the website through `doc_path`, relative to the project root. Migration leaves `output` out of `octavo.yml` when that destination is sufficient.

An explicit Quarto `output-dir` can name a different destination. Migration preserves that override, expressing it relative to the notebooks directory for Octavo. For example, a project-level `website` directory becomes `../website` when notebooks live in `nbs`.

In [ ]:
#| export
def _octavo_output(nbs_path, qy, ny):
    "Return a Quarto output override relative to notebooks, or None when doc_path suffices"
    out_dir = (ny.get('project', {}) or {}).get('output-dir') or (qy.get('project', {}) or {}).get('output-dir')
    if not out_dir: return
    cfg = get_config(nbs_path)
    out_dir = (cfg.config_path/Path(out_dir).expanduser()).resolve()
    if out_dir == cfg.doc_path.resolve(): return
    return os.path.relpath(out_dir, Path(nbs_path).resolve()).replace(os.sep, '/')

Octavo derives its sidebar from directories, which matches the sidebar nbdev generates. A project with `custom_sidebar` writes `sidebar.yml` by hand instead, often grouping pages from one directory into sections. Migration converts that file into `sidebar-links`: pages become links and sections become groups. Entries without text take the page's title at build time. Quarto's `auto` entries and glob patterns have no Octavo equivalent; migration reports and omits them.

In [ ]:
#| export
def _octavo_sidebar(nbs_path):
    "Convert the project's custom sidebar when enabled"
    p = Path(nbs_path)/'sidebar.yml'
    if not get_config(nbs_path).custom_sidebar or not p.exists(): return {}
    sb = nested_idx(yaml.safe_load(p.read_text()) or {}, 'website', 'sidebar') or {}
    return quarto_sidebar(sb, p)

Octavo uses Quarto's names and meanings for the settings the two share, but keeps them in one flat `octavo.yml`, where Quarto nests them under `website` and `format: html`. `_create_octavo_cfg` copies those settings across, converts the navigation bar and a hand-written sidebar, and prints the other Quarto settings, which Octavo has no equivalent for. An output override is included only when the Quarto destination differs from nbdev’s `doc_path`. The migration command writes the result to `octavo.yml` only when that file does not already exist.

This is not a complete translation of Quarto configuration. The reported settings still need manual review.

In [ ]:
#| export
def _create_octavo_cfg(nbs_path):
    "Convert Quarto settings with nbdev's overlay, output path and custom sidebar"
    nbs_path = Path(nbs_path).resolve()
    qy,ny = _quarto_config(nbs_path)
    output = _octavo_output(nbs_path, qy, ny)
    cfg = quarto_config(qy, nbs_path, overrides=ny) | _octavo_sidebar(nbs_path)
    if output is not None: cfg['output'] = output
    return cfg

Render scripts keep their executable body. Only the leading YAML docstring needs the same metadata conversion as a document.

In [ ]:
#| export
_script_fm = re.compile(r'\A(?P<q>"{3}|\'{3})---\n(?P<fm>.*?)\n---(?P=q)', re.DOTALL)

def _script_md(m):
    "Frontmatter block from a `_script_fm` match on a render script"
    return f"---\n{m.group('fm')}\n---\n"

def repl_script_frontmatter_keys(s):
    "Normalize a render script's leading YAML docstring for Octavo"
    if not (m := _script_fm.match(s)): return s
    old = _script_md(m)
    new = repl_frontmatter_keys(old)
    if new == old: return s
    return m.group('q') + new.rstrip('\n') + m.group('q') + s[m.end():]

Switching builders requires source changes as well as new configuration: renamed documents need updated links, and Quarto-specific markup needs conversion. Migration checks rename conflicts and configuration before editing, preserves an existing `octavo.yml`, and retains `_quarto.yml` for comparison. Changes are made in place without automatic rollback, so commit or back up the project first.

In [ ]:
#| export
NBDEV_KEYS = {'skip_exec', 'skip_showdoc'}

def _migrate_octavo(path):
    "Migrate documents and nbdev render scripts using the project's configuration"
    path = Path(path).expanduser()
    scripts = source_files(path).map(lambda o: path/o).filter(lambda p: p.name.endswith('.qmd.py'))
    script_moves = {p:p.with_name(p.name.removesuffix('.qmd.py')+'.md.py') for p in scripts}
    moves = migrate_quarto(path, config=_create_octavo_cfg, extra_keys=NBDEV_KEYS, extra_moves=script_moves)
    for p in script_moves.values():
        migrate_md_file(p, [repl_script_frontmatter_keys])
        if m := _script_fm.match(p.read_text()):
            report_page_keys(p, extra_keys=NBDEV_KEYS, metadata=frontmatter(_script_md(m))[0])
    return moves

A destination collision must stop migration before an existing file can be overwritten.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d)
    (p/'page.qmd.py').touch()
    (p/'page.md.py').touch()
    with expect_fail(FileExistsError, contains='page.md.py'): _migrate_octavo(p)

The default comes from the project configuration, so the command also works when notebooks live outside `nbs`.

In [ ]:
#| export
@call_parse
def nbdev_migrate_octavo(
    path:str=None, # Documentation directory (defaults to configured nbs_path)
):
    "Migrate Quarto-era nbdev project to Octavo"
    moves = _migrate_octavo(get_config().nbs_path if path is None else path)
    for old,new in moves.items(): print(f'{old} -> {new}')

A documentation project can mix written pages, notebooks, and pages generated by Python scripts. The following project includes all three. After migration, we inspect their filenames, metadata, links, and retained content.

In [ ]:
migration_tmp = tempfile.TemporaryDirectory()
d = migration_tmp.name
nbs = Path(d)/'nbs'
(Path(d)/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "nbs"\ncustom_sidebar = true\n')
(nbs/'blog/posts').mkdir(parents=True)
(nbs/'images').mkdir()
(nbs/'guide.qmd').write_text(r'''---
subtitle: A handy guide
toc-depth: 2
custom-key: keep-this
---

See [blog](blog/index.qmd). The `.qmd` format is mentioned here.

::: {.content-visible when-format="html" style="text-align: center"}
<iframe src="video"></iframe>
:::
''')
(nbs/'blog/index.qmd').write_text(r'''---
title: Blog
subtitle: All posts
sidebar: false
listing:
  categories: true
---

Back to [guide](../guide.qmd).
''')
(nbs/'llms.txt').write_text('- [API](api.html.md): reference\n- [Guide](guide.md)\n')

52

Render scripts carry metadata in a Python docstring. Their program body and non-Markdown generated assets must remain unchanged.

Only `.qmd.py` scripts need migration to `.md.py`. Existing `.md.py` pages and other double-suffix generators, such as `.svg.py` and `.html.py`, keep their filenames and contents. Ordinary `.py` files are unchanged too.

In [ ]:
(nbs/'index.qmd.py').write_text(r'''"""---
title: Home
subtitle: Write, test, document, and distribute software packages and technical articles — all in one place, your notebook.
page-layout: custom
toc: false
---"""
print("# Home")
''')
(nbs/'blog/posts/render.qmd.py').write_text('"""---\ntitle: Rendered\ntoc-depth: 3\n---"""\nprint("# Rendered")\n')
(nbs/'images/circles.svg.py').write_text('"""---\n---"""\nprint("<svg/>")\n')

30

In [ ]:
for name in ('existing.md.py', 'images/widget.html.py', 'ordinary.py'):
    (nbs/name).write_text('raise RuntimeError("migration must not execute scripts")\n')
script_before = {p:p.read_text() for p in nbs.rglob('*.py')}

This project includes both supported and unsupported navbar icons. Migration should report the unsupported entry without losing the supported links. The project sets `custom_sidebar`, and its hand-written `sidebar.yml` has a section and an `auto` entry, which migration reports.

In [ ]:
(nbs/'_quarto.yml').write_text(r'''website:
  title: demo
  favicon: favicon.png
  description: A demo site
  open-graph: true
  twitter-card:
    site: "@demo"
    image: https://x.y/card.png
    image-width: 1200
  navbar:
    left:
      - text: Guide
        href: guide.qmd
    right:
      - icon: github
        href: https://github.com/x/y
      - icon: twitter
        href: https://twitter.com/x
        aria-label: Follow us
      - icon: bug
        href: https://x.y/issues
format:
  html:
    css: styles.css
    toc-depth: 4
''')
(nbs/'styles.css').write_text('body { color: red; }')
(nbs/'sidebar.yml').write_text(r'''website:
  sidebar:
    contents:
      - guide.qmd
      - section: More
        contents:
          - blog/index.qmd
          - 01_extra.ipynb
          - auto
''')

163

Notebook frontmatter and format-specific Markdown need the same conversion as standalone documents. Most notebooks have no frontmatter at all, such as `api.ipynb` here.

In [ ]:
nb = new_nb([mk_cell(r'''---
summary: Notebook summary
categories: [tutorial]
toc-depth: 2
---

[guide](guide.qmd) and [blog](blog/index.qmd)

::: {.content-visible when-format="markdown"}
![CI](badge.svg)
:::''', 'markdown')])
write_nb(nb, nbs/'index.ipynb')
write_nb(new_nb([mk_cell('# API\n\nSee the [guide](guide.qmd).', 'markdown')]), nbs/'api.ipynb')

Renaming includes nested documents and Markdown render scripts. Other generated asset types retain their filenames and content.

In [ ]:
quarto_before = (nbs/'_quarto.yml').read_bytes()
with redirect_stdout(StringIO()) as out: moves = _migrate_octavo(nbs)
test_eq({str(k.relative_to(nbs)):str(v.relative_to(nbs)) for k,v in moves.items()}, {
    'blog/index.qmd':'blog/index.md', 'blog/posts/render.qmd.py':'blog/posts/render.md.py',
    'guide.qmd':'guide.md', 'index.qmd.py':'index.md.py'})
assert not (nbs/'guide.qmd').exists() and (nbs/'guide.md').exists()
assert not (nbs/'blog/index.qmd').exists() and (nbs/'blog/index.md').exists()
assert not (nbs/'index.qmd.py').exists() and (nbs/'index.md.py').exists()
assert not (nbs/'blog/posts/render.qmd.py').exists() and (nbs/'blog/posts/render.md.py').exists()
test_eq((nbs/'images/circles.svg.py').read_text(), '"""---\n---"""\nprint("<svg/>")\n')

In [ ]:
for old,text in script_before.items():
    new = moves.get(old, old)
    actual = new.read_text()
    if old not in moves: test_eq(actual, text)
    else: test_eq(actual[_script_fm.match(actual).end():], text[_script_fm.match(text).end():])

Render script metadata keeps Quarto's names, such as `page-layout` and `toc`, which Octavo reads as they are. Its `subtitle` becomes the description, and a long description must survive YAML serialization without inserted line continuations.

In [ ]:
idx_txt = (nbs/'index.md.py').read_text()
idx_fm = frontmatter(_script_md(_script_fm.match(idx_txt)))[0]
test_eq(idx_fm['page-layout'], 'custom')
test_eq(idx_fm['toc'], False)
assert 'subtitle' not in idx_fm and '— all in one place' in idx_fm['description']
assert 'technical\\\n' not in idx_txt

The converted configuration retains the site identity and supported navigation. The sidebar keeps its order and section; `01_extra.ipynb` links to its published name, `extra.html`. The site description, Open Graph and Twitter card settings, and `toc-depth`, carry over unchanged. It sets `math: dollars`, so `$…$` stays inline maths as in Quarto. Quarto CSS is not enabled in `octavo.yml`. The original configuration and stylesheet remain unchanged for manual adaptation.

In [ ]:
test_eq((nbs/'_quarto.yml').read_bytes(), quarto_before)
assert (nbs/'octavo.yml').exists()
cfg = yaml.safe_load((nbs/'octavo.yml').read_text())
test_eq(cfg['title'], 'demo')
test_eq(cfg['math'], 'dollars')
test_eq(cfg['description'], 'A demo site')
test_eq(cfg['open-graph'], True)
test_eq(cfg['twitter-card'], {'site': '@demo', 'image': 'https://x.y/card.png', 'image-width': 1200})
assert 'output' not in cfg
test_eq(cfg['toc-depth'], 4)
assert 'css' not in cfg
test_eq((nbs/'styles.css').read_text(), 'body { color: red; }')
test_eq(cfg['favicon'], 'favicon.png')
test_eq(cfg['nav-links'], [{'text': 'Guide', 'href': 'guide.html'}])
test_eq(cfg['tools'], [{'icon': 'github', 'href': 'https://github.com/x/y', 'label': 'GitHub'},
                       {'icon': 'x', 'href': 'https://twitter.com/x', 'label': 'Follow us'}])
test_eq(cfg['sidebar-links'], [{'href': 'guide.html'},
                               {'text': 'More', 'children': [{'href': 'blog/index.html'}, {'href': 'extra.html'}]}])

Migration printed what Octavo will ignore, for manual review: Quarto's stylesheet setting, the unbundled icon and the `auto` sidebar entry, the guide's `custom-key`, the blog listing's `categories` filter option:

In [ ]:
report = out.getvalue()
assert 'format.html.css' in report and "'bug'" in report and "'auto'" in report
assert f"{nbs/'guide.md'}: Octavo ignores custom-key" in report
assert f"{nbs/'blog/index.md'}: Octavo ignores listing.categories" in report
print(report)

/private/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/octavo.yml: dropped tool icon 'bug' (https://x.y/issues): not bundled by octavo. Add it back with `icon: <path-or-url>` to your own SVG.
/private/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs: Octavo has no equivalent for format.html.css
/private/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/sidebar.yml: dropped sidebar entry 'auto': Octavo has no equivalent. List those pages explicitly.
/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/blog/index.md: Octavo ignores listing.categories
/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/guide.md: Octavo ignores custom-key



In [ ]:
out

<_io.StringIO>

Frontmatter aliases change at the top level. Relative links still need to reach the renamed document from its own directory, and `llms.txt` now links Octavo's Markdown copies.

In [ ]:
guide_fm,guide_body = frontmatter((nbs/'guide.md').read_text())
test_eq(guide_fm['description'], 'A handy guide')
assert 'subtitle' not in guide_fm
test_eq(guide_fm['toc-depth'], 2)
test_eq(guide_fm['custom-key'], 'keep-this')

blog_fm,blog_body = frontmatter((nbs/'blog/index.md').read_text())
test_eq(blog_fm['description'], 'All posts')
test_eq(blog_fm['sidebar'], False)
test_eq(blog_body.strip(), 'Back to [guide](../guide.md).')
test_eq((nbs/'llms.txt').read_text(), '- [API](api.md): reference\n- [Guide](guide.md)\n')

Notebook Markdown receives the same metadata conversion. A second migration must leave the converted metadata unchanged.

In [ ]:
nb_fm,nb_body = frontmatter(read_nb(nbs/'index.ipynb').cells[0].source)
test_eq(nb_fm['description'], 'Notebook summary')
test_eq(nb_fm['categories'], ['tutorial'])
assert 'summary' not in nb_fm
test_eq(nb_fm['toc-depth'], 2)

test_eq(_migrate_octavo(nbs), {})
test_eq(frontmatter((nbs/'guide.md').read_text())[0], guide_fm)
migration_tmp.cleanup()

/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/blog/index.md: Octavo ignores listing.categories
/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmp26a6l0uo/nbs/guide.md: Octavo ignores custom-key


This project keeps notebooks in `docs/notebooks` and sets `doc_path` to `public`. Migration omits redundant output settings; the build still writes to the project’s `public` directory. A different Quarto destination is preserved.

In [ ]:
nested_tmp = tempfile.TemporaryDirectory()
d = nested_tmp.name
project = Path(d)
nbs = project/'docs/notebooks'
nbs.mkdir(parents=True)
(project/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "docs/notebooks"\ndoc_path = "public"\n')
(nbs/'_quarto.yml').write_text('website:\n  title: Demo\nformat:\n  html:\n    toc: false\n')
(nbs/'guide.qmd').write_text('# Guide')

7

A project with nested notebooks should still publish to its configured project-level output directory. The installed CLI must find that directory without a `--path` override.

Migration does not invent footer text. A project must supply its own footer.

In [ ]:
with working_directory(project): run('nbdev-migrate-octavo')
settings = octavo_config(nbs)
assert 'output' not in settings
test_eq(settings['toc'], False)
assert 'footer' not in settings
assert (nbs/'guide.md').exists() and (nbs/'_quarto.yml').exists()
with working_directory(project): run('nbdev-docs')
assert (project/'public/guide.html').exists()

test_eq(_octavo_output(nbs, {'project': {'output-dir': 'public'}}, {}), None)
test_eq(_octavo_output(nbs, {'project': {'output-dir': 'website'}}, {}), '../../website')
test_eq(_octavo_output(nbs, {'project': {'output-dir': 'website'}}, {'project': {'output-dir': 'public'}}), None)
test_eq(_octavo_output(nbs, {'project': {'output-dir': str(project/'elsewhere')}}, {}), '../../elsewhere')

An existing Octavo configuration belongs to the author. Repeating migration must preserve it rather than regenerate defaults.

In [ ]:
before = (nbs/'octavo.yml').read_text()
test_eq(_migrate_octavo(nbs), {})
test_eq((nbs/'octavo.yml').read_text(), before)
nested_tmp.cleanup()